# 10 — Everything together

Run this last. It loads the results every other notebook saved and puts them in one place.

Nothing new is measured here.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, harness, report
import pandas as pd

df = harness.load_all()
df = df[df["duckdb_s"].notna()].copy()
print(f"Loaded {len(df)} operations from {df['notebook'].nunique()} notebooks.")
print(C.summary())

## The headline

In [ ]:
report.headline(df)

## Every operation, ranked by how much the gap varies

**This chart is the actual point of the project.**

Not "DuckDB is faster" — that is a boring claim and an easy one to argue with. The finding is
that **the gap varies enormously depending on what the job does**. Operations near the bottom
of this chart are where a distributed engine earns its overhead. Operations near the top are
where it is being paid for and not used.

That is a far more useful thing to hand a team than a single number.

In [ ]:
report.ratio_chart(df)

In [ ]:
report.category_chart(df)

## Every result

In [ ]:
pd.set_option("display.max_rows", 100)
display(report.results_table(df).sort_values("Category"))

## How much of this was the same SQL?

In [ ]:
same = int(df["identical_sql"].sum()); total = len(df)
print(f"   {same} of {total} operations ran on a BYTE-IDENTICAL SQL string in both engines.\n")
diff = df[~df["identical_sql"]][["id", "operation", "category"]]
if len(diff):
    print("   The exceptions, and why:")
    for _, r in diff.iterrows():
        print(f"      {r['id']:>3}  {r['operation']}")
    print()
print("   This matters more than any timing on this page.")
print("   It means moving a job from one engine to the other is not a rewrite:")
print("   the SQL stays the same and the Parquet files stay the same.")
print()
print('   Which undercuts the usual objection -- "but what if it grows later?"')
print("   If it grows past what one machine handles, you move it. That is an afternoon,")
print("   not a migration project. So there is little to gain from paying for a cluster")
print("   up front on jobs that do not need one yet.")

## Did the answers match?

In [ ]:
ok   = int((df["same_answer"] == True).sum())
bad  = df[df["same_answer"] == False]
appr = int(df["same_answer"].isna().sum())
print(f"   Identical answers : {ok}")
print(f"   Not compared      : {appr}  (approximate algorithms, or write jobs)")
print(f"   DIFFERENT         : {len(bad)}")
if len(bad):
    display(bad[["id", "operation", "category"]])
else:
    print("\n   Nothing differed. So the only thing that varied was how long it took.")

## The rule of thumb

Two questions decide it, and neither of them is "is our data big?"

**How much data does this job touch, and how many of these run at once?**

| Reach for **DuckDB** when... | Stay with **Spark / Databricks** when... |
|---|---|
| The job reads gigabytes, not terabytes | The job reads terabytes |
| One job or one person at a time | Many people sharing one cluster |
| Someone is exploring, in a notebook | Streaming or live data |
| A scheduled transform or quality check | Two enormous tables joined together |
| Tests and pipeline checks | You need the catalog, permissions and governance |

**What this project does not show**, stated plainly so nobody has to ask:

* nothing about *our* data, *our* cluster or *our* costs — the data here is invented
* nothing above the size a single machine holds, which is where a cluster wins and should
* nothing about governance, sharing, streaming or access control
* Spark's real strength, which is many machines. Every number here came from one.

**The honest summary:** this is not "replace Spark". It is that a job's size and shape should
decide the tool, and that assuming *big* by default has a measurable cost on the jobs that are
not.

In [ ]:
print("=" * 78)
print("  THE SHORT VERSION".center(78))
print("=" * 78)
cmp = df[df["spark_s"].notna()]
print(f"""
  Where           : one ordinary machine -- {C.plural(C.CORES, 'thread')}, {C.RAM_GB:.0f} GB RAM
  Fairness        : both engines given {C.ENGINE_MEMORY_MB} MB and {C.plural(C.ENGINE_THREADS, 'thread')}
  Data            : {C.human(C.MAIN_SIZE)} synthetic sales rows, 22 columns, defects planted on purpose
  Operations      : {len(df)} across {df['category'].nunique()} kinds of work
  Same SQL        : {int(df['identical_sql'].sum())} of {len(df)} ran byte-identical in both engines
  Answers matched : {int((df['same_answer'] == True).sum())} confirmed identical, 0 different
""")
if len(cmp):
    print(f"""  Median gap      : {cmp['ratio'].median():.0f}x
  Range           : {cmp['ratio'].min():g}x to {cmp['ratio'].max():g}x

  The finding is the RANGE, not the median. How much a single machine wins by
  depends on what the job actually does -- which is the thing worth checking
  before assuming any job needs a cluster.
""")
print("=" * 78)